# Notebook 02 — Análise de Qualidade por Sistema e Decaimento Temporal

**Desafio**: Cientista de Dados Pleno — Squad WhatsApp | Prefeitura do Rio de Janeiro

## Objetivos

**Parte 1.1** — Correlacionar cada sistema de origem com a performance real dos disparos, corrigindo o viés de seleção.

**Parte 1.2** — Investigar o "decaimento temporal": o tempo desde a última atualização do telefone em um sistema impacta a taxa de entrega? Existe um prazo de validade para um dado ser considerado "quente"?

---

## 0. Setup e Carregamento

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
from scipy import stats
from scipy.optimize import curve_fit
import gcsfs

plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
sns.set_palette('Set2')

BUCKET = 'gs://case_vagas/whatsapp'

print('Ambiente configurado.')

In [ ]:
fs = gcsfs.GCSFileSystem(token='anon')

with fs.open(f"{BUCKET.replace('gs://', '')}/base_disparo_mascarado") as f:
    df_disparo = pd.read_parquet(f)

with fs.open(f"{BUCKET.replace('gs://', '')}/dim_telefone_mascarado") as f:
    df_tel = pd.read_parquet(f)

print(f'Disparos carregados: {len(df_disparo):,}')
print(f'Telefones carregados: {len(df_tel):,}')

---

# Parte 1.1 — Taxas de Entrega por Sistema de Origem

## 1. Construção do Dataset Analítico

Para correlacionar **sistema de origem** com **performance de disparo**, precisamos:

1. Fazer join entre `base_disparo` e `dim_telefone` via chave de telefone
2. Explodir o array `telefone_aparicoes` para obter uma linha por (disparo × sistema de origem)
3. Isso nos diz: "este disparo foi para um número que consta no sistema X"

> **Nota metodológica**: Um telefone pode aparecer em N sistemas. Ao explodir, cada disparo é replicado N vezes. Estamos perguntando: "Dispatches para números que constam no sistema X têm qual taxa de entrega?"

In [ ]:
# Flag de sucesso (DELIVERED ou READ = entregue com sucesso)
df_disparo['sucesso'] = df_disparo['status_disparo'].isin(['DELIVERED', 'READ']).astype(int)

# Join com dimensão de telefones
df_joined = df_disparo[['contato_telefone', 'status_disparo', 'sucesso', 'criacao_envio_datahora']].merge(
    df_tel[['telefone_mascarado', 'telefone_aparicoes', 'telefone_tipo', 'telefone_qualidade']],
    left_on='contato_telefone',
    right_on='telefone_mascarado',
    how='inner'
)

print(f'Disparos após join: {len(df_joined):,} ({len(df_joined)/len(df_disparo)*100:.1f}% do total)')
print(f'Taxa de sucesso geral no subset: {df_joined["sucesso"].mean()*100:.2f}%')

In [ ]:
# Explodir array de aparições: 1 linha por (disparo × sistema)
df_exp = df_joined.explode('telefone_aparicoes').reset_index(drop=True)

# Normalizar structs do array para colunas
aparicoes_norm = pd.json_normalize(df_exp['telefone_aparicoes'])
df_exp = pd.concat([
    df_exp[['contato_telefone', 'status_disparo', 'sucesso', 'criacao_envio_datahora',
            'telefone_tipo', 'telefone_qualidade']].reset_index(drop=True),
    aparicoes_norm
], axis=1)

# Converter datas
df_exp['criacao_envio_datahora'] = pd.to_datetime(df_exp['criacao_envio_datahora'])
df_exp['registro_data_atualizacao'] = pd.to_datetime(df_exp['registro_data_atualizacao'])

# Calcular idade do dado no momento do disparo
df_exp['dias_desde_atualizacao'] = (
    df_exp['criacao_envio_datahora'] - df_exp['registro_data_atualizacao']
).dt.days

print(f'Dataset analítico: {len(df_exp):,} linhas (disparo × sistema)')
df_exp.head(3)

## 2. Taxas de Entrega por Sistema — Análise Bruta

In [ ]:
# Agregação por sistema de origem
agg_sistema = df_exp.groupby('id_sistema').agg(
    total_aparicoes=('sucesso', 'count'),
    total_sucesso=('sucesso', 'sum')
).reset_index()

agg_sistema['taxa_entrega_bruta'] = agg_sistema['total_sucesso'] / agg_sistema['total_aparicoes']
agg_sistema = agg_sistema.sort_values('taxa_entrega_bruta', ascending=False)

print('Taxa de entrega bruta por sistema:\n')
print(agg_sistema.to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Volume de aparições
agg_plot = agg_sistema.sort_values('total_aparicoes', ascending=False)
agg_plot['total_aparicoes'].plot(kind='bar', ax=axes[0], color='#3498db', edgecolor='white',
                                  xlabel='Sistema')
axes[0].set_title('Volume de Aparições por Sistema\n(Reflete uso histórico, não qualidade)', fontweight='bold')
axes[0].set_xticklabels(agg_plot['id_sistema'], rotation=45, ha='right')
axes[0].set_ylabel('Nº de Aparições')
axes[0].yaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f'{x/1e3:.0f}k'))

# Taxa bruta por sistema (ordenada por taxa)
agg_taxa = agg_sistema.sort_values('taxa_entrega_bruta', ascending=False)
axes[1].bar(range(len(agg_taxa)), agg_taxa['taxa_entrega_bruta'] * 100,
            color='#2ecc71', edgecolor='white')
axes[1].set_title('Taxa de Entrega Bruta por Sistema\n(Sem correção de viés de seleção)', fontweight='bold')
axes[1].set_xticks(range(len(agg_taxa)))
axes[1].set_xticklabels(agg_taxa['id_sistema'], rotation=45, ha='right')
axes[1].set_ylabel('Taxa de Entrega (%)')
axes[1].yaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f'{x:.0f}%'))

for i, (_, row) in enumerate(agg_taxa.iterrows()):
    axes[1].text(i, row['taxa_entrega_bruta'] * 100 + 0.5,
                 f"{row['taxa_entrega_bruta']*100:.1f}%", ha='center', fontsize=9)

plt.suptitle('Análise Bruta: Volume vs Taxa de Entrega por Sistema', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 3. Identificação e Correção do Viés de Seleção

### O Problema

O enunciado avisa: *"Algumas bases de dados já são consideradas 'mais quentes' e aparecem com maior frequência nos logs"*. Isso cria um **viés de seleção**: sistemas preferidos historicamente acumulam mais registros, mas isso não prova que são melhores.

O scatter abaixo evidencia se volume e taxa estão correlacionados (o que indicaria viés).

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

ax.scatter(
    agg_sistema['total_aparicoes'],
    agg_sistema['taxa_entrega_bruta'] * 100,
    s=agg_sistema['total_aparicoes'] / agg_sistema['total_aparicoes'].max() * 2000 + 50,
    alpha=0.7, color='#3498db', edgecolors='white', linewidth=1.5
)

for _, row in agg_sistema.iterrows():
    ax.annotate(row['id_sistema'],
                (row['total_aparicoes'], row['taxa_entrega_bruta'] * 100),
                textcoords='offset points', xytext=(8, 4), fontsize=9)

# Taxa média geral como referência
taxa_media = df_joined['sucesso'].mean() * 100
ax.axhline(taxa_media, color='#e74c3c', linestyle='--', alpha=0.7, label=f'Média geral: {taxa_media:.1f}%')

ax.set_xlabel('Volume de Aparições (escala log)', fontsize=11)
ax.set_ylabel('Taxa de Entrega Bruta (%)', fontsize=11)
ax.set_title('Viés de Seleção: Volume vs Taxa de Entrega\n'
             'Sistemas com alto volume não são necessariamente os melhores',
             fontweight='bold', fontsize=12)
ax.set_xscale('log')
ax.yaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f'{x:.0f}%'))
ax.legend()

plt.tight_layout()
plt.show()

# Correlação estatística entre log-volume e taxa
corr, pval = stats.pearsonr(np.log(agg_sistema['total_aparicoes']), agg_sistema['taxa_entrega_bruta'])
print(f'Correlação de Pearson (log-volume × taxa): r = {corr:.3f}  (p-valor = {pval:.4f})')
if abs(corr) > 0.3 and pval < 0.05:
    print('Correlação significativa detectada: viés de seleção presente. Correção necessária.')
else:
    print('Baixa correlação entre volume e taxa — sistemas foram usados de forma relativamente equilibrada.')

## 4. Correção: Wilson Score Lower Bound

### Por que Wilson Score?

Para sistemas com **poucos registros**, a taxa bruta pode ser enganosa:
- Sistema A: 9/10 entregas → 90% — mas com apenas 10 casos, pode ser sorte
- Sistema B: 850/1.000 entregas → 85% — muito mais confiável

O **Wilson Score Lower Bound** fornece o limite inferior do IC de 95%: *"com 95% de confiança, a taxa real é pelo menos este valor"*. Penaliza sistemas com pouca evidência.

$$\text{Wilson LB} = \frac{\hat{p} + \frac{z^2}{2n} - z\sqrt{\frac{\hat{p}(1-\hat{p})}{n} + \frac{z^2}{4n^2}}}{1 + \frac{z^2}{n}}$$

In [ ]:
def wilson_lower_bound(successos, total, z=1.96):
    """Wilson Score Lower Bound para proporções binomiais."""
    if total == 0:
        return 0.0
    p_hat = successos / total
    denominador = 1 + z**2 / total
    centro = p_hat + z**2 / (2 * total)
    margem = z * np.sqrt(p_hat * (1 - p_hat) / total + z**2 / (4 * total**2))
    return (centro - margem) / denominador


def wilson_upper_bound(successos, total, z=1.96):
    if total == 0:
        return 1.0
    p_hat = successos / total
    denominador = 1 + z**2 / total
    centro = p_hat + z**2 / (2 * total)
    margem = z * np.sqrt(p_hat * (1 - p_hat) / total + z**2 / (4 * total**2))
    return (centro + margem) / denominador


# Aplicar ao dataset de sistemas
agg_sistema['wilson_lb'] = agg_sistema.apply(
    lambda r: wilson_lower_bound(r['total_sucesso'], r['total_aparicoes']), axis=1
)
agg_sistema['wilson_ub'] = agg_sistema.apply(
    lambda r: wilson_upper_bound(r['total_sucesso'], r['total_aparicoes']), axis=1
)

# Score final = Wilson Lower Bound (conservador, corrigido para volume)
agg_sistema['score_sistema'] = agg_sistema['wilson_lb']
agg_sistema = agg_sistema.sort_values('score_sistema', ascending=False).reset_index(drop=True)
agg_sistema.index += 1  # ranking começa em 1

print('Ranking de Sistemas — Score Corrigido (Wilson Lower Bound IC 95%):\n')
display_cols = ['id_sistema', 'total_aparicoes', 'total_sucesso',
                'taxa_entrega_bruta', 'wilson_lb', 'wilson_ub', 'score_sistema']
print(agg_sistema[display_cols].to_string(float_format='{:.4f}'.format))

In [ ]:
fig, ax = plt.subplots(figsize=(13, 6))

agg_plot2 = agg_sistema.reset_index(drop=True).sort_values('score_sistema', ascending=False)
sistemas = agg_plot2['id_sistema'].tolist()
x = np.arange(len(sistemas))
taxas = agg_plot2['taxa_entrega_bruta'].values * 100
lb = agg_plot2['wilson_lb'].values * 100
ub = agg_plot2['wilson_ub'].values * 100
scores = agg_plot2['score_sistema'].values * 100

# Taxa bruta
ax.bar(x - 0.2, taxas, width=0.35, label='Taxa Bruta', color='#3498db', alpha=0.7, edgecolor='white')
# Score Wilson LB
ax.bar(x + 0.2, scores, width=0.35, label='Wilson LB (Score Ranking)', color='#e74c3c', alpha=0.85, edgecolor='white')

# IC 95%
ax.errorbar(x - 0.2, taxas, yerr=[taxas - lb, ub - taxas],
            fmt='none', color='#2c3e50', capsize=4, linewidth=1.5, label='IC 95%')

ax.set_xticks(x)
ax.set_xticklabels(sistemas, rotation=45, ha='right', fontsize=10)
ax.set_ylabel('Taxa de Entrega (%)')
ax.set_title('Taxa Bruta vs Score Corrigido (Wilson LB) por Sistema\n'
             'A diferença revela o impacto do viés de seleção',
             fontweight='bold')
ax.yaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f'{x:.0f}%'))
ax.legend()
ax.axhline(taxa_media, color='gray', linestyle=':', alpha=0.7, label='Média geral')

plt.tight_layout()
plt.show()

### Conclusão da Parte 1.1

O ranking por **Wilson Lower Bound** é mais justo que a taxa bruta:
- Sistemas com alto volume e alta taxa: score próximo da taxa bruta (evidência sólida)
- Sistemas com baixo volume: score penalizado (incerteza alta)

Isso garante que **não premiamos sistemas pouco testados** nem **punimos sistemas sólidos pelo uso intenso**.

---